# Clean the Crime Rate LGA dataset and insert it into DB

This notebook builds a data pipeline for the pooled-normalised LGA crime rate CSV and writes cleaned data to PostgreSQL.

## Cleaning rules
- read the CSV with pandas
- drop completely empty rows
- trim and normalise whitespace in text fields
- parse the ISO-8601 date string and extract the **financial year end** as an integer (e.g. `2013-06-01` → `2013`)
- rename columns to match the DB schema
- keep `rate` as NULL where the source value is missing (suppressed data); do **not** coerce to 0
- round `rate` to 2 decimal places
- remove exact duplicate rows

In [9]:
from pathlib import Path
import pandas as pd

INPUT_CSV = Path("input/crime_rate_pooled_normalised_lga_2024_ann.csv")

INPUT_CSV

PosixPath('input/crime_rate_pooled_normalised_lga_2024_ann.csv')

In [10]:
df_raw = pd.read_csv(INPUT_CSV)
df_raw.columns

Index(['frequency', 'date', 'measure', 'region_type', 'region', 'offence_type',
       'offence', 'value'],
      dtype='str')

In [15]:
def clean_crime_rate(input_path: Path):
    # Read the CSV.
    df = pd.read_csv(input_path)

    # Drop completely empty rows.
    df = df.dropna(axis=0, how="all").reset_index(drop=True)

    # Clean up all text columns: normalise whitespace and convert blanks to missing values.
    object_cols = df.select_dtypes(include="object").columns
    for col in object_cols:
        s = df[col].astype("string")
        s = s.str.replace(r"\s+", " ", regex=True).str.strip()
        s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        df[col] = s

    # Parse the ISO-8601 date and extract the financial-year-end as an integer.
    # e.g. '2013-06-01T00:00:00Z' → 2013
    df["year"] = (
        pd.to_datetime(df["date"], utc=True)
        .dt.year
        .astype("Int64")
    )

    # Rename columns to match the DB schema.
    column_mapping = {
        "region":       "lga_name",
        "offence":      "offence",
        "value":        "adjusted_rate",
        "frequency":    "frequency",
        "measure":      "measure",
        "region_type":  "region_type",
        "offence_type": "offence_type",
    }
    df = df.rename(columns=column_mapping)

    # Round adjusted_rate to 2 decimal places; keep NULL where data is suppressed.
    df["adjusted_rate"] = pd.to_numeric(df["adjusted_rate"], errors="coerce").round(2)

    # Drop the original date column — year now carries the temporal dimension.
    df = df.drop(columns=["date"])

    # Remove exact duplicate rows.
    rows_before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    duplicates_removed = rows_before - len(df)

    summary = {
        "input_rows":        rows_before,
        "output_rows":       len(df),
        "duplicates_removed": duplicates_removed,
        "null_rates":        int(df["adjusted_rate"].isna().sum()),
        "lgas":              df["lga_name"].nunique(),
        "years":             sorted(df["year"].dropna().unique().tolist()),
        "offences":          sorted(df["offence"].dropna().unique().tolist()),
    }
    return df, summary

In [16]:
clean_df, summary = clean_crime_rate(INPUT_CSV)
summary

/var/folders/w4/l4h21lrj0d59d23sfzvj3t5m0000gn/T/ipykernel_73700/1533378089.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include="object").columns


{'input_rows': 13893,
 'output_rows': 13893,
 'duplicates_removed': 0,
 'null_rates': 318,
 'lgas': 508,
 'years': [2013,
  2014,
  2015,
  2016,
  2017,
  2018,
  2019,
  2020,
  2021,
  2022,
  2023,
  2024],
 'offences': ['dwelling', 'motor_vehicle', 'violent']}

In [5]:
clean_df.head()

,frequency,measure,region_type,lga_name,offence_type,offence,rate,year
0,ANNUAL,OFFENCE_RATE_POOLED_NORMALISED,LGA_2024,Albury,OFF,dwelling,1753.32,2013
1,ANNUAL,OFFENCE_RATE_POOLED_NORMALISED,LGA_2024,Armidale,OFF,dwelling,1557.27,2013
2,ANNUAL,OFFENCE_RATE_POOLED_NORMALISED,LGA_2024,Ballina,OFF,dwelling,1012.85,2013
3,ANNUAL,OFFENCE_RATE_POOLED_NORMALISED,LGA_2024,Balranald,OFF,dwelling,1629.27,2013
4,ANNUAL,OFFENCE_RATE_POOLED_NORMALISED,LGA_2024,Bathurst,OFF,dwelling,1465.14,2013


In [17]:
print(f"Blank rows remaining:           {int(clean_df.isna().all(axis=1).sum())}")
print(f"Exact duplicates remaining:     {int(clean_df.duplicated().sum())}")
print(f"Rows violating unique key:      {int(clean_df.duplicated(subset=['lga_name','year','offence']).sum())}")
text_cols = clean_df.select_dtypes(include="string").columns
ws_cells = sum(clean_df[col].fillna('').str.fullmatch(r'\s*').sum() for col in text_cols)
print(f"Whitespace-only text cells:     {int(ws_cells)}")

Blank rows remaining:           0
Exact duplicates remaining:     0
Rows violating unique key:      0
Whitespace-only text cells:     0


# Insert Clean Data into DB

In [18]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# -----------------------------
# 1. DATABASE CONNECTION
# -----------------------------
load_dotenv('.env.db', override=True)

DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = os.getenv("DB_PORT")
DB_NAME     = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

# -----------------------------
# 2. CREATE TABLE FROM SCHEMA
# -----------------------------
schema_path = Path("schemas/crime_rate_lga.sql")

with open(schema_path, "r", encoding="utf-8") as f:
    schema_sql = f.read()

with engine.begin() as conn:
    conn.execute(text(schema_sql))

print("Schema created or already exists.")
print(DB_NAME)

Schema created or already exists.
carelink


In [20]:
# -----------------------------
# 3. PREPARE DATAFRAME
# -----------------------------
df_insert = clean_df.copy()

target_columns = [
    "lga_name",
    "year",
    "offence",
    "adjusted_rate",
    "frequency",
    "measure",
    "region_type",
    "offence_type",
]

df_insert = df_insert[target_columns].copy()

# Text columns
text_cols = ["lga_name", "offence", "frequency", "measure", "region_type", "offence_type"]
for col in text_cols:
    df_insert[col] = df_insert[col].astype("string").str.strip()

# Integer columns
df_insert["year"] = pd.to_numeric(df_insert["year"], errors="coerce").astype("Int64")

# Numeric columns
df_insert["adjusted_rate"] = pd.to_numeric(df_insert["adjusted_rate"], errors="coerce")

In [21]:
# -----------------------------
# 4. INSERT INTO DATABASE
# -----------------------------
df_insert.to_sql(
    "crime_rate_lga",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)

print(f"Inserted {len(df_insert)} rows into crime_rate_lga")

# -----------------------------
# 5. VALIDATE
# -----------------------------
with engine.begin() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM crime_rate_lga"))
    row_count = result.scalar()

print("Rows in table:", row_count)

Inserted 13893 rows into crime_rate_lga
Rows in table: 13893
